In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
from typing import List, Dict, Optional
from pathlib import Path



In [ ]:
lon_min, lat_min, lon_max, lat_max = 9.8, 50.2, 12.7, 51.7 

In [13]:
# this doewnloads the weather data of the water measuring spots

BASE_URL = "https://kshww2.thueringen.de/FROST-Server/v1.1"

def _get(path: str, params: dict | None = None) -> dict:
    """Your exact _get function"""
    url = f"{BASE_URL}/{path.lstrip('/')}"
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_station_ids_in_bbox(lon_min: float, lat_min: float, 
                             lon_max: float, lat_max: float) -> List[Dict]:
    """Step 1: Get station IDs and names in bounding box"""
    polygon = (
        f"POLYGON(({lon_min} {lat_min}, {lon_max} {lat_min}, "
        f"{lon_max} {lat_max}, {lon_min} {lat_max}, {lon_min} {lat_min}))"
    )
    data = _get(
        "Things",
        {
            "$filter": f"st_within(Locations/location,geography'{polygon}')",
            "$select": "name,description,@iot.id,properties",
            "$top": 200,
            "$orderby": "name asc",
        },
    )
    return [{"id": t["@iot.id"], "name": t["name"]} for t in data.get("value", [])]


def get_station_coords(thing_id: int) -> Optional[Dict[str, float]]:
    """Step 2: Get lat/lon for a single station"""
    data = _get(
        f"Things({thing_id})/Locations",
        {"$select": "location", "$top": 1}
    )
    locations = data.get("value", [])
    if locations:
        coords = locations[0].get("location", {}).get("coordinates", [])
        if len(coords) >= 2:
            return {"lat": coords[1], "lon": coords[0]}
    return None


def fetch_open_meteo(lat: float, lon: float, start: str, end: str, 
                     max_retries: int = 3) -> Optional[pd.DataFrame]:
    """Step 3: Fetch hourly weather from Open-Meteo with retries"""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "hourly": [
            "temperature_2m", "relative_humidity_2m", "dew_point_2m",
            "precipitation", "surface_pressure", "cloud_cover",
            "wind_speed_10m", "wind_direction_10m", "shortwave_radiation"
        ],
        "timezone": "auto"
    }
    
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            df = pd.DataFrame(data["hourly"])
            df["time"] = pd.to_datetime(df["time"])
            return df.set_index("time")
        except Exception as e:
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 10
                print(f"      Retry {attempt+1}/{max_retries} in {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"      FAILED after {max_retries} attempts: {e}")
                return None


def build_weather_dataset(bbox: tuple, start: str, end: str, 
                          cache_dir: str = "weather_cache") -> pd.DataFrame:
    """
    Orchestrator with:
    - Station-by-station caching (resume if interrupted)
    - Better error handling
    - Progress saving
    """
    lon_min, lat_min, lon_max, lat_max = bbox
    
    # Create cache directory
    Path(cache_dir).mkdir(exist_ok=True)
    
    # Get stations
    stations = get_station_ids_in_bbox(lon_min, lat_min, lon_max, lat_max)
    print(f"Found {len(stations)} stations in bbox\n")
    
    all_dfs = []
    failed = []
    
    for i, station in enumerate(stations):
        print(f"[{i+1}/{len(stations)}] {station['name']}", end="")
        
        # Check if already cached
        cache_file = Path(cache_dir) / f"station_{station['id']}.csv"
        if cache_file.exists():
            print(" - loading from cache")
            df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
        else:
            coords = get_station_coords(station["id"])
            
            if not coords:
                print(" - SKIP (no coordinates)")
                failed.append(station)
                continue
            
            print(f" ({coords['lat']:.4f}, {coords['lon']:.4f})", end="")
            
            df = fetch_open_meteo(coords["lat"], coords["lon"], start, end)
            
            if df is None:
                print(" - FAILED")
                failed.append(station)
                continue
            
            # Save to cache
            df.to_csv(cache_file)
            print(" - fetched & cached")
        
        df["station_id"] = station["id"]
        df["station_name"] = station["name"]
        all_dfs.append(df)
        
        # Rate limiting
        time.sleep(2)
    
    # Report
    print(f"\n{'='*50}")
    print(f"Complete: {len(all_dfs)}/{len(stations)} stations fetched")
    if failed:
        print(f"Failed stations:")
        for s in failed:
            print(f"  - {s['name']} (ID: {s['id']})")
    
    if all_dfs:
        return pd.concat(all_dfs)
    else:
        raise RuntimeError("No data fetched!")


# Usage
if __name__ == "__main__":
    weather = build_weather_dataset(
        bbox=(9.8, 50.2, 12.7, 51.7),
        start="2025-01-01",
        end="2026-01-01",
        cache_dir="thuringia_weather_cache"
    )
    weather.to_csv("thuringia_weather_2025.csv")
    print(f"\nSaved {len(weather):,} records")

Found 174 stations in bbox

[1/174] Almerswind-Grümpen - loading from cache
[2/174] Almerswind-Itz - loading from cache
[3/174] Ammern - loading from cache
[4/174] Apfelstädtaue - loading from cache
[5/174] Arenshausen - loading from cache
[6/174] Arnstadt - loading from cache
[7/174] Artern - loading from cache
[8/174] Bad Blankenburg (alt) - loading from cache
[9/174] Bellstedt (alt) - loading from cache
[10/174] Benshausen - loading from cache
[11/174] Berga - loading from cache
[12/174] Bernterode - loading from cache
[13/174] Blankenburg-Weidmannsheil (alt) - loading from cache
[14/174] Blankenstein (alt) - loading from cache
[15/174] Blankenstein-Rosenthal - loading from cache
[16/174] Bleicherode - loading from cache
[17/174] Bottendorf - loading from cache
[18/174] Breitungen - loading from cache
[19/174] Bruchstedt (alt) - loading from cache
[20/174] Buttelstedt - loading from cache
[21/174] Büßleben - loading from cache
[22/174] Camburg-Stöben - loading from cache
[23/174] Do

KeyboardInterrupt: 